# 11 — Function Calling Training

Train the model to correctly invoke `aro ask` tools (function calling).
ARO Coder has 18 tools available via `aro ask`. This notebook generates
training pairs that teach the model:

1. **When** to call each tool (trigger recognition)
2. **How** to format the call (correct JSON arguments)
3. **What** to do with the result (interpret + next step)
4. **Fix patterns** — `aro_check` error → `edit_file` fix → verify

**Input:** Tool definitions from `../../Sources/AROAsk/Tools/`
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl` (appended)

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')

import json, re, random
from pathlib import Path
from collections import Counter

DATA_OUT = DATA_ROOT / '11_function_calling'
DATA_OUT.mkdir(parents=True, exist_ok=True)
print(f'Output: {DATA_OUT}')

## Tool definitions and example calls

In [ ]:
TOOLS = [
    {'name': 'read_file', 'params': ['path', 'offset?', 'limit?'], 'examples': [
        {'user': 'Read the main.aro file', 'args': {'path': 'main.aro'}},
        {'user': 'Show me lines 10-20 of users.aro', 'args': {'path': 'users.aro', 'offset': 10, 'limit': 10}},
    ]},
    {'name': 'write_file', 'params': ['path', 'content'], 'examples': [
        {'user': 'Create a hello world ARO app', 'args': {'path': 'main.aro', 'content': '(Application-Start: Hello) {\n    Log "Hello!" to the <console>.\n    Return an <OK: status> for the <greeting>.\n}'}},
    ]},
    {'name': 'edit_file', 'params': ['path', 'old_string', 'new_string'], 'examples': [
        {'user': 'Fix the string concat in main.aro', 'args': {'path': 'main.aro', 'old_string': '<first> + " " + <last>', 'new_string': '<first> ++ " " ++ <last>'}},
    ]},
    {'name': 'list_dir', 'params': ['path?'], 'examples': [
        {'user': 'What files are in this project?', 'args': {}},
    ]},
    {'name': 'grep', 'params': ['pattern', 'path?', 'glob?'], 'examples': [
        {'user': 'Find all feature sets that use Emit', 'args': {'pattern': 'Emit', 'glob': '*.aro'}},
    ]},
    {'name': 'run_shell', 'params': ['command'], 'examples': [
        {'user': 'Show the git log', 'args': {'command': 'git log --oneline -5'}},
    ]},
    {'name': 'aro_check', 'params': ['path'], 'examples': [
        {'user': 'Check if my app has syntax errors', 'args': {'path': '.'}},
        {'user': 'Validate the code', 'args': {'path': '.'}},
    ]},
    {'name': 'aro_run', 'params': ['path', 'args?'], 'examples': [
        {'user': 'Run the app', 'args': {'path': '.'}},
    ]},
    {'name': 'aro_test', 'params': ['path'], 'examples': [
        {'user': 'Run the tests', 'args': {'path': '.'}},
    ]},
    {'name': 'aro_build', 'params': ['path'], 'examples': [
        {'user': 'Build a native binary', 'args': {'path': '.'}},
    ]},
    {'name': 'parse_aro', 'params': ['path'], 'examples': [
        {'user': 'Show me the AST', 'args': {'path': 'main.aro'}},
    ]},
    {'name': 'list_actions', 'params': [], 'examples': [
        {'user': 'What actions can I use?', 'args': {}},
    ]},
    {'name': 'create_plugin', 'params': ['name', 'language', 'handle?'], 'examples': [
        {'user': 'Create a Swift plugin for analytics', 'args': {'name': 'analytics', 'language': 'swift', 'handle': 'Analytics'}},
    ]},
    {'name': 'write_openapi', 'params': ['content'], 'examples': [
        {'user': 'Generate an OpenAPI contract for my user API', 'args': {'content': 'openapi: 3.0.3\ninfo:\n  title: User API\n  version: 1.0.0'}},
    ]},
    {'name': 'generate_docs', 'params': ['path'], 'examples': [
        {'user': 'Generate docs for my app', 'args': {'path': '.'}},
    ]},
    {'name': 'list_proposals', 'params': [], 'examples': [
        {'user': 'What language proposals exist?', 'args': {}},
    ]},
    {'name': 'read_proposal', 'params': ['number'], 'examples': [
        {'user': 'Show me the events proposal', 'args': {'number': '0007'}},
    ]},
    {'name': 'search_project', 'params': ['query', 'k?'], 'examples': [
        {'user': 'Find code related to authentication', 'args': {'query': 'user authentication login'}},
    ]},
]
print(f'{len(TOOLS)} tools, {sum(len(t["examples"]) for t in TOOLS)} examples')


## Generate training pairs

Three pair types:
1. **Direct call** — user asks → model calls the right tool
2. **Fix chains** — read → check → edit → verify (the `/fix` workflow)
3. **Tool selection** — given a task, which tool to use?

In [ ]:
import json as _json

def tool_call(name, args):
    return f'<tool_call>{_json.dumps({"name": name, "arguments": args})}</tool_call>'

pairs = []

# ---- Type 1: Direct tool calls ----
for tool in TOOLS:
    for ex in tool['examples']:
        pairs.append({
            'instruction': ex['user'],
            'output': tool_call(tool['name'], ex['args']),
            'source': f'fc_direct:{tool["name"]}',
            'task_type': 'function_calling',
            'score': 1.0,
        })
print(f'Direct calls: {len(pairs)}')

# ---- Type 2: Fix chains (aro ask /fix workflow) ----
FIX_SCENARIOS = [
    {'code': '(Application-Start: Demo) {\n    Compute the <msg> from "Hello" + <name>.\n    Log <msg> to the <console>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:45: error: Expected \'\'>\', but got +',
     'old': '"Hello" + <name>', 'new': '"Hello, " ++ <name>', 'desc': 'string concat uses ++'},
    {'code': '(Application-Start: Demo) {\n    Create the <count> with 0.\n    Compute the <count> from <count> + 1.\n    Return an <OK: status> for the <result>.\n}',
     'error': '3:18: error: Cannot rebind variable \'count\'',
     'old': 'Compute the <count>', 'new': 'Compute the <incremented>', 'desc': 'immutable variable'},
    {'code': '(Application-Start: Demo) {\n    Compute the <is-valid> from <age> > 18.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:22: error: Reserved prefix \'is-\'',
     'old': '<is-valid>', 'new': '<valid>', 'desc': 'reserved prefix'},
    {'code': '(Application-Start: Demo) {\n    Retrieve the <user> with the <user-repo>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:30: error: Expected preposition \'from\', but got \'with\'',
     'old': 'Retrieve the <user> with', 'new': 'Retrieve the <user> from', 'desc': 'wrong preposition'},
    {'code': '(Application-Start: Demo) {\n    Store the <user> from the <user-repo>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:24: error: Expected preposition \'into\', but got \'from\'',
     'old': 'Store the <user> from', 'new': 'Store the <user> into', 'desc': 'Store uses into'},
    {'code': '(Application-Start: Demo) {\n    Log <msg> for the <console>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:18: error: Expected preposition \'to\', but got \'for\'',
     'old': 'Log <msg> for the', 'new': 'Log <msg> to the', 'desc': 'Log uses to'},
    {'code': '(Application-Start: Demo) {\n    Emit the <user> to the <event>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:10: error: Expected event qualifier',
     'old': 'Emit the <user> to the <event>.', 'new': 'Emit a <UserCreated: event> with <user>.', 'desc': 'Emit syntax'},
    {'code': '(Application-Start: Demo) {\n    Extract the <id> from the <request: id>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:40: error: Use pathParameters for URL params',
     'old': '<request: id>', 'new': '<pathParameters: id>', 'desc': 'path params'},
]

for s in FIX_SCENARIOS:
    # Full chain: user asks to fix -> read -> check -> edit -> verify
    # Step 1: user asks, model reads file
    pairs.append({'instruction': 'Fix the errors in main.aro',
                  'output': tool_call('read_file', {'path': 'main.aro'}),
                  'source': 'fc_fix:read', 'task_type': 'function_calling', 'score': 1.0})
    # Step 2: model sees code, runs check
    pairs.append({'instruction': f'File content:\n```aro\n{s["code"]}\n```\nCheck for errors.',
                  'output': tool_call('aro_check', {'path': '.'}),
                  'source': 'fc_fix:check', 'task_type': 'function_calling', 'score': 1.0})
    # Step 3: model sees error, fixes with edit
    pairs.append({'instruction': f'aro_check error:\n{s["error"]}\nFix it.',
                  'output': tool_call('edit_file', {'path': 'main.aro', 'old_string': s['old'], 'new_string': s['new']}),
                  'source': 'fc_fix:edit', 'task_type': 'function_calling', 'score': 1.0})
    # Step 4: verify
    pairs.append({'instruction': f'Fixed {s["desc"]}. Verify.',
                  'output': tool_call('aro_check', {'path': '.'}),
                  'source': 'fc_fix:verify', 'task_type': 'function_calling', 'score': 1.0})

print(f'Fix chains: {len(FIX_SCENARIOS) * 4}')

# ---- Type 3: Tool selection (which tool for this task?) ----
TOOL_SELECTION = [
    ('I want to see what files exist', 'list_dir'),
    ('Check my code for errors', 'aro_check'),
    ('Run my application', 'aro_run'),
    ('Find where the user model is defined', 'grep'),
    ('Create a new Rust plugin', 'create_plugin'),
    ('Generate API documentation', 'generate_docs'),
    ('Build a native binary of my app', 'aro_build'),
    ('What ARO actions are available?', 'list_actions'),
    ('Read the control flow proposal', 'read_proposal'),
    ('Search for authentication-related code', 'search_project'),
    ('Run the test suite', 'aro_test'),
    ('Show me the parse tree', 'parse_aro'),
    ('Write an openapi.yaml for my API', 'write_openapi'),
    ('Replace the old function name with the new one', 'edit_file'),
    ('Execute a git command', 'run_shell'),
]
for user_q, tool_name in TOOL_SELECTION:
    tool = next(t for t in TOOLS if t['name'] == tool_name)
    args = tool['examples'][0]['args'] if tool['examples'] else {}
    pairs.append({
        'instruction': user_q,
        'output': tool_call(tool_name, args),
        'source': f'fc_select:{tool_name}',
        'task_type': 'function_calling',
        'score': 1.0,
    })

print(f'Tool selection: {len(TOOL_SELECTION)}')
print(f'Total: {len(pairs)} function calling pairs')


## Save and merge

In [ ]:
clean_notebook_pairs('NB11')
for p in pairs:
    p['notebook'] = 'NB11'
new_count = save_notebook_pairs('NB11', pairs)
print(f'Saved {new_count} function calling pairs to {PAIRS_FILE}')

own_file = DATA_OUT / 'function_calling_pairs.jsonl'
with open(own_file, 'w') as f:
    for p in pairs:
        f.write(json.dumps(p) + '\n')
print(f'Local copy: {own_file}')


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white', 'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa', 'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa', 'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '11_function_calling.png'

_tool_counts = Counter()
_type_counts = Counter()
for p in pairs:
    src = p.get('source', '')
    tool = src.split(':')[1] if ':' in src else 'other'
    _tool_counts[tool] += 1
    ptype = src.split(':')[0].replace('fc_', '')
    _type_counts[ptype] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

_sorted = sorted(_tool_counts.items(), key=lambda x: -x[1])
_names = [t for t, _ in _sorted]
_vals = [c for _, c in _sorted]
x = np.arange(len(_names))
ax1.bar(x, _vals, color='#3498db', edgecolor='white', width=0.7)
for i, v in enumerate(_vals):
    ax1.text(i, v + 0.2, str(v), ha='center', fontsize=8, fontweight='bold')
ax1.set_xticks(list(x))
ax1.set_xticklabels(_names, rotation=45, ha='right', fontsize=7)
ax1.set_ylabel('Pairs')
ax1.set_title('Pairs per tool', fontsize=11, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

_ts = sorted(_type_counts.items(), key=lambda x: -x[1])
ax2.bar([t for t,_ in _ts], [c for _,c in _ts],
        color=['#2ecc71','#3498db','#e67e22','#9b59b6'][:len(_ts)], edgecolor='white')
for i, (t, c) in enumerate(_ts):
    ax2.text(i, c + 0.2, str(c), ha='center', fontsize=9, fontweight='bold')
ax2.set_ylabel('Pairs')
ax2.set_title('Pairs by type', fontsize=11, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

fig.suptitle(f'Function Calling Training \u2014 {len(pairs)} pairs, {len(TOOLS)} tools',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')


In [ ]:
import csv
from datetime import date as _date
_run_dir = Path('.') / 'run' / _date.today().isoformat()
csv_path = _run_dir / '11_function_calling.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['tool', 'pair_type', 'instruction_preview', 'output_preview'])
    for p in pairs:
        src = p.get('source', '')
        tool = src.split(':')[1] if ':' in src else 'other'
        ptype = src.split(':')[0].replace('fc_', '')
        w.writerow([tool, ptype, p['instruction'][:150], p['output'][:200]])
print(f'CSV: {csv_path} ({len(pairs)} rows)')
